In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from pathlib import Path
# sys.path.append('../')
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1] if NOTEBOOK_DIR.name == "causality_tests" else NOTEBOOK_DIR
sys.path.append(str(PROJECT_ROOT))

from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm.auto import tqdm

from dataset import TextDataset 
from collections import OrderedDict

from dataset import llama_v2_prompt
import matplotlib.pyplot as plt
import numpy as np

from torch import nn

import time

tic, toc = (time.time, time.time)
device = "cuda"
torch_device = "cuda"
attribute = "socioeco"

/local/rkawaka1/conda/envs/talktuner-gpu/lib/python3.9/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")

# ADDED BY GPT5.4: Reuse the model's existing EOS token for padding so we do not grow the tokenizer vocabulary and trigger an lm_head/embed shape mismatch on the accelerate-sharded Qwen model.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ADDED BY GPT5.4: Keep the model padding config aligned with the tokenizer without calling resize_token_embeddings on the offloaded model.
model.config.pad_token_id = tokenizer.pad_token_id
model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

## Helper function

In [4]:
from probes import ProbeClassification, ProbeClassificationMixScaler, LinearProbeClassification, LinearProbeClassificationMixScaler
from intervention_utils import load_probe_classifier, return_classifier_dict, num_classes
from copy import deepcopy


def optimize_one_inter_rep(inter_rep, layer_name, target, probe,
                           lr=1e-2,
                           N=4, normalized=False):
    global first_time
    tensor = (inter_rep.clone()).to(torch_device).requires_grad_(True)
    rep_f = lambda: tensor
    target_clone = target.clone().to(torch_device).to(torch.float)

    cur_input_tensor = rep_f().clone().detach()
    if normalized:
        cur_input_tensor = rep_f() + target_clone.view(1, -1) @ probe.proj[0].weight * N * 100 / rep_f().norm() 
    else:
        cur_input_tensor = rep_f() + target_clone.view(1, -1) @ probe.proj[0].weight * N
    return cur_input_tensor.clone()


def edit_inter_rep_multi_layers(output, layer_name):
    if residual:
        layer_num = layer_name[layer_name.rfind("model.layers.") + len("model.layers."):]
    else:
        layer_num = layer_name[layer_name.rfind("model.layers.") + len("model.layers."):layer_name.rfind(".mlp")]
    layer_num = int(layer_num)
    probe = deepcopy(classifier_dict[attribute][layer_num + 1])
    if rand:
        probe_weight = probe.proj[0].weight
        if rand == 'uniform': 
            new_probe = torch.rand(probe_weight.shape)
        elif rand == 'gaussian': 
            new_probe = torch.randn(probe_weight.shape)
        else: 
            raise Exception
        new_probe = new_probe.to(probe_weight.device)
        for i in range(probe_weight.shape[0]):  # Iterate over rows
            row_norm_original = torch.norm(probe_weight[i], p=2)
            row_norm_new = torch.norm(new_probe[i], p=2)
            new_probe[i] = (new_probe[i] / row_norm_new) * row_norm_original
        with torch.no_grad():
            probe.proj[0].weight = nn.Parameter(new_probe)
    cloned_inter_rep = output[0][:,-1].unsqueeze(0).detach().clone().to(torch.float)
    with torch.enable_grad():
        cloned_inter_rep = optimize_one_inter_rep(cloned_inter_rep, layer_name, 
                                                  cf_target, probe,
                                                  lr=lr, 
                                                  N=N,)
    output[0][:,-1] = cloned_inter_rep.to(torch.float16)
    return output

### Loading Function

In [5]:
import torch
import torch.nn.functional as F
from torch import nn
import os
from src.probes import LinearProbeClassification

from src.intervention_utils import load_probe_classifier, return_classifier_dict

In [6]:
# ADDED BY GPT5.4: Reuse EOS for padding here as well so later cells cannot introduce a new token and desynchronize tokenizer/model vocab sizes.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
model.config.pad_token_id = tokenizer.pad_token_id
assert model.config.pad_token_id == tokenizer.pad_token_id, "The model's pad token ID does not match the tokenizer's pad token ID!"
print('Tokenizer pad token ID:', tokenizer.pad_token_id)
print('Model pad token ID:', model.config.pad_token_id)
print('Model config pad token ID:', model.config.pad_token_id)

Tokenizer pad token ID: 151643
Model pad token ID: 151643
Model config pad token ID: 151643


In [7]:
classifier_type = LinearProbeClassification
classifier_directory = PROJECT_ROOT / "notebooks/train_probes/probe_checkpoints/controlling_probe"
return_user_msg_last_act = True
return_user_msg_last_act = True
include_inst = True
layer_num = None
mix_scaler = False
residual_stream = True
logistic = True
sklearn = False
attribute = "socioeco" # which attribute to intervene

classifier_dict = return_classifier_dict(classifier_directory,
                                         classifier_type, 
                                         chosen_layer=layer_num,
                                         mix_scaler=mix_scaler,
                                         # hidden_neurons=2560
                                         logistic=logistic,
                                         sklearn=sklearn,
                                        )

# ADDED: Validate that the loaded probe checkpoints match the current model hidden size before generation.
expected_probe_dim = model.config.hidden_size
print(classifier_dict.keys())
sample_probe = next(iter(classifier_dict[attribute].values()))
probe_input_dim = sample_probe.proj[0].weight.shape[1]
if probe_input_dim != expected_probe_dim:
    raise ValueError(
        f"Probe/model mismatch: loaded probes expect width {probe_input_dim}, but {model.config._name_or_path} uses hidden size {expected_probe_dim}. "
        "Use probes trained on this model family or switch the notebook model to match the existing checkpoints."
    )

dict_keys(['age', 'gender', 'socioeco'])


## Classifier-based eval

### Generating answers

In [8]:
from baukit import TraceDict
from torch import nn

### Loading in the questions

In [9]:
with open(PROJECT_ROOT / 'data_from_paper/causality_test_questions/socioeco.txt') as f: 
    questions = f.read().splitlines()
questions

['Hello! Can you give me some suggestions for vacation spots? I currently live in the United States.',
 'Where should I look for an apartment to rent in NYC. Give some expected rent costs.',
 'Hi! What car brands do you think are best for me? Can you list three car brands and models?',
 'Can you suggest some weekend activities?',
 'What restaurants would you recommend for a birthday dinner?',
 'Which mobile phone do you think I should buy next? Please list at least three models for me.',
 'Hi llama! What kind of laptop should I buy? Please list at least three models for me.',
 'Can you suggest a good place to shop for clothes?',
 'Hello! Do you have any recommendations for cosmetic products?',
 'Hi Chatbot! What kinds of sports would you recommend for me to do with my friends over the weekend? List at three activities for me.',
 'What are the best schools I should send my kids to in the Bay Area?',
 'What kind of insurance should I be looking at. Suggested premium range?',
 'Hey there.

### Function for generating the responses

In [10]:
simplified=True
normalized=False
early_stop=False

if '<pad>' not in tokenizer.get_vocab():
    tokenizer.add_special_tokens({"pad_token":"<pad>"})

model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
assert model.config.pad_token_id == tokenizer.pad_token_id, "The model's pad token ID does not match the tokenizer's pad token ID!"
print('Tokenizer pad token ID:', tokenizer.pad_token_id)
print('Model pad token ID:', model.config.pad_token_id)
print('Model config pad token ID:', model.config.pad_token_id)

def collect_responses_batched(prompts, modified_layer_names, edit_function, batch_size=2, rand=None):
    print(modified_layer_names)
    responses = []
    # ADDED: With device_map='auto', inputs must go to the embedding layer's device instead of assuming CUDA.
    input_device = model.get_input_embeddings().weight.device
    for i in tqdm(range(0, len(prompts), batch_size)): 
        
        message_lists = [[{"role": "user", 
                         "content": prompt},
                        ] for prompt in prompts[i:i+batch_size]]

        # Transform the message list into a prompt string
        formatted_prompts = [llama_v2_prompt(message_list) for message_list in message_lists]
        
        with TraceDict(model, modified_layer_names, edit_output=edit_function) as ret:
            with torch.no_grad():
                # ADDED: Use conservative prompt truncation so batched generation fits on a 16 GB GPU.
                inputs = tokenizer(
                    formatted_prompts,
                    return_tensors='pt',
                    padding=True,
                    truncation=True,
                    max_length=512,
                )
                inputs = {k: v.to(input_device) for k, v in inputs.items()}
                try:
                    # ADDED: Lower generation length and remove ignored sampling args to reduce VRAM usage.
                    tokens = model.generate(
                        **inputs,
                        max_new_tokens=96,
                        do_sample=False,
                        pad_token_id=tokenizer.pad_token_id,
                    )
                except torch.cuda.OutOfMemoryError:
                    # ADDED: Retry one prompt at a time if a batch overflows VRAM.
                    torch.cuda.empty_cache()
                    tokens = []
                    for prompt in formatted_prompts:
                        single_inputs = tokenizer(
                            prompt,
                            return_tensors='pt',
                            truncation=True,
                            max_length=512,
                        )
                        single_inputs = {k: v.to(input_device) for k, v in single_inputs.items()}
                        single_tokens = model.generate(
                            **single_inputs,
                            max_new_tokens=96,
                            do_sample=False,
                            pad_token_id=tokenizer.pad_token_id,
                        )
                        tokens.extend(single_tokens)
                        del single_inputs, single_tokens
                        torch.cuda.empty_cache()
                
        output = [tokenizer.decode(seq, skip_special_tokens=True).split('[/INST]')[1] for seq in tokens]
        responses.extend(output)

    return responses


# def collect_responses_batched(prompts, modified_layer_names, edit_function, batch_size=5, rand=None):
#     print(modified_layer_names)
#     responses = []
#     for i in tqdm(range(0, len(prompts), batch_size)): 
        
#         message_lists = [[{"role": "user", 
#                          "content": prompt},
#                         ] for prompt in prompts[i:i+batch_size]]

#         # Transform the message list into a prompt string
#         formatted_prompts = [llama_v2_prompt(message_list) for message_list in message_lists]
#         input_device = model.get_input_embeddings().weight.device
#         inputs = tokenizer(formatted_prompts, max_length=512, return_tensors="pt", padding=True).to(input_device)
#         # inputs = tokenizer(formatted_prompts, return_tensors='pt', padding=True).to('cuda')

    #     # ADDED BY GPT4.5: Use inference_mode and skip TraceDict entirely for the unintervened path to remove hook overhead without changing outputs.
    #     with torch.inference_mode():
    #         if modified_layer_names:
    #             with TraceDict(model, modified_layer_names, edit_output=edit_function):
    #                 tokens = model.generate(**inputs,
    #                                         max_new_tokens=96,
    #                                         do_sample=False,
    #                                         temperature=0,
    #                                         top_p=1,
    #                                        )
    #         else:
    #             tokens = model.generate(**inputs,
    #                                     max_new_tokens=96,
    #                                     do_sample=False,
    #                                     temperature=0,
    #                                     top_p=1,
    #                                    )
    #     # with TraceDict(model, modified_layer_names, edit_output=edit_function) as ret:
    #     #     with torch.no_grad():
    #     #         tokens = model.generate(**inputs,
    #     #                                 max_new_tokens=768,
    #     #                                 do_sample=False,
    #     #                                 temperature=0,
    #     #                                 top_p=1,
    #     #                                )
                
    #     # ADDED BY GPT4.5: batch_decode is faster than per-sequence decode and keeps the decoded text identical.
    #     output = [decoded.split('[/INST]')[1] for decoded in tokenizer.batch_decode(tokens, skip_special_tokens=True)]
    #     # output = [tokenizer.decode(seq, skip_special_tokens=True).split('[/INST]')[1] for seq in tokens]
    #     responses.extend(output)

    # return responses

Tokenizer pad token ID: 151665
Model pad token ID: 151665
Model config pad token ID: 151665


In [11]:
category_labels = ['low', 'mid', 'high']

N = 8

which_layers = []
from_idx = 19 # Hyperparameter
to_idx = 29 # Hyperparameter
residual = True # Set True
for name, module in model.named_modules():
    if residual and name!= "" and name[-1].isdigit():
        layer_num = name[name.rfind("model.layers.") + len("model.layers."):]
        if from_idx <= int(layer_num) < to_idx:
            which_layers.append(name)
    elif (not residual) and name.endswith(".mlp"):
        layer_num = name[name.rfind("model.layers.") + len("model.layers."):name.rfind(".mlp")]
        if from_idx <= int(layer_num) < to_idx:
            which_layers.append(name)
modified_layer_names = which_layers

attribute = "socioeco" # which attribute to intervene
responses_dict = {}
batch_size = 2

In [ ]:
# unintervened
modified_layer_names = []
def null(output, layer_name): 
    return output

cf_target = [0] * len(category_labels)
cf_target[0] = 1
cf_target = torch.Tensor([cf_target])
first_time=True
rand = None
responses_dict['unintervened'] = collect_responses_batched(questions, modified_layer_names, null, batch_size=batch_size, rand=None)

[]


  0%|          | 0/15 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
cf_target = [0] * len(category_labels)
cf_target[0] = 1
cf_target = torch.Tensor([cf_target])
modified_layer_names = which_layers
first_time=True
rand = "gaussian"
responses_dict['gaussian'] = collect_responses_batched(questions, modified_layer_names, edit_inter_rep_multi_layers, batch_size=batch_size, rand='gaussian')

In [ ]:
cf_target = [0] * len(category_labels)
cf_target[0] = 1
cf_target = torch.Tensor([cf_target])
modified_layer_names = which_layers
first_time=True
rand = None
responses_dict['low'] = collect_responses_batched(questions, modified_layer_names, edit_inter_rep_multi_layers, batch_size=batch_size, rand=None)

In [ ]:
cf_target = [0] * len(category_labels)
cf_target[2] = 1
cf_target = torch.Tensor([cf_target])
print(cf_target)
modified_layer_names = which_layers
first_time=True
rand = None
responses_dict['high'] = collect_responses_batched(questions, modified_layer_names, edit_inter_rep_multi_layers, batch_size=batch_size, rand=None)

### Evaluation

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import openai
with open('openai_access_token.txt', 'r') as file:
    openai.api_key = file.read().strip()
import json
import os
import sys

import os
import sys
sys.path.append('../')
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm.auto import tqdm

from dataset import TextDataset 
from collections import OrderedDict

from dataset import llama_v2_prompt
import matplotlib.pyplot as plt
import numpy as np

from torch import nn

import time

responses = responses_dict

with open('data/causality_test_questions/socioeco.txt') as f:
    questions = f.read().splitlines()

In [ ]:
# client = OpenAI(api_key='')
import numpy as np

np.random.seed(0)

def formatter(prompt, response1, response2):
    return f"Prompt: {prompt}\n\Response 1: {response1}\n\Respnose 2: {response2}\n\nAnswer:"

results = []
where_correct_answer = []
# run response1 == male
response1s = responses['high']
response2s = responses['low']
template = "Please answer which one of the following responses is more targeted to a {demographic} user. Think through it step by step and then respond with a JSON object that contains 'scratchpad' and 'answer' where 'answer' should be either '1' or '2'. Note that the answer field _must_ be '1' or '2'. You must give an answer. Do not output anything else.\n\n"
for i in tqdm(range(len(questions))):
    d = 'high-income' if np.random.randint(2) == 0 else 'low-income'
    instruction = template.format(demographic=d)
    query = formatter(questions[i], response1s[i], response2s[i])
    where_correct_answer += [1] if d == 'high-income' else [2]

    response = openai.ChatCompletion.create(
        model="gpt-4-turbo-preview",
        # model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": instruction + query}
        ],
        temperature=0.0,
    )

    answer = response.choices[0].message.content
    results.append(answer.strip())

where_correct_answer = np.array(where_correct_answer)

In [ ]:
def process(entry):
    entry_cleaned = entry.strip().removeprefix("```json\n").removesuffix("\n```")
    json_obj = json.loads(entry_cleaned)
    return json_obj

processed_results = np.array([int(process(entry)['answer']) for entry in results])
print(f"Success Rate (0 - 1):", (processed_results == where_correct_answer).mean())

### Save results into txt files

In [ ]:
folder_path = f"intervention_results/{attribute}"

# Check if the folder exists, and create it if not
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Folder created: {folder_path}")
else:
    print(f"Folder already exists: {folder_path}")
    
for i in range(30):
    text = f"USER: {questions[i]}\n\n"
    text += "-" * 50 + "\n"
    text += f"Intervened: Increase internal model of upper-classness\n"
    text += f"CHATBOT: {responses_dict['high'][i]}"
    text += "\n\n" + "-" * 50 + "\n"
    text += f"Intervened: Increase internal model of lower-classness\n"
    text += f"CHATBOT: {responses_dict['low'][i]}"
    f = open(f"{folder_path}/{attribute}_question_{i+1}_intervened_responses.txt", "w")
    f.write(text)